# Image classification on [Cats-vs-Dogs](https://www.kaggle.com/competitions/dogs-vs-cats) dataset — *PyTorch version*

_Image classification models in a data-poor context_

---

This tutorial is highly inspired by the [blog](https://blog.keras.io/building-powerful-image-classification-models-using-very-little-data.html) from [François Chollet](https://fchollet.com/) at the initiative of [`Keras`](https://keras.io/). This notebook is a **PyTorch translation** of the original `TensorFlow`/`Keras` tutorial.

In this tutorial you will learn to :
* Use convolutional networks to build image classifiers on color images
* Use pre-trained models ($\texttt{VGG}$, $\texttt{Inception}$, _etc._) to improve the accuracy of the results
* Fine-Tune pre-trained models

In this tutorial, we focus on the (seemingly) simple problem of recognizing dogs and cats in images.

<br>

> **Keras $\to$ PyTorch cheat sheet**
>
> | Keras / TensorFlow | PyTorch |
> |---|---|
> | `ImageDataGenerator` + `flow_from_dataframe` | `torch.utils.data.Dataset` + `DataLoader` + `torchvision.transforms` |
> | `Sequential([...])`, `model.add(...)` | `nn.Sequential(...)` or a `nn.Module` subclass |
> | `Conv2D`, `MaxPooling2D`, `Dense`, `Flatten`, `Dropout` | `nn.Conv2d`, `nn.MaxPool2d`, `nn.Linear`, `nn.Flatten`, `nn.Dropout` |
> | `GlobalAveragePooling2D` | `nn.AdaptiveAvgPool2d(1)` + `nn.Flatten()` |
> | activation as an argument (`activation='relu'`) | activation as a **layer** (`nn.ReLU()`) |
> | `model.compile(loss=..., optimizer=...)` | a `criterion` object + an `optimizer` object |
> | `model.fit(...)` | an explicit **training loop** (written below once and reused) |
> | `model.evaluate(...)` / `model.predict(...)` | a loop under `torch.no_grad()` |
> | `layer.trainable = False` | `for p in layer.parameters(): p.requires_grad = False` |
> | tensors are `(N, H, W, C)` | tensors are `(N, C, H, W)` |


## Libraries

In [ ]:
# Utils
import os
import shutil
import time

# Maths - Stats
import numpy as np
import pandas as pd
import random as rd
from PIL import Image

# Data visualization
from matplotlib import pyplot as plt
import seaborn as sns

# Deep Learning Librairies
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import transforms
from torchvision.models import vgg16, VGG16_Weights

print("torch:", torch.__version__, "| torchvision:", torchvision.__version__)

These code lines allow you to check if your computer is using CPU or GPU ressources. <br>
**Warning** : You won't be able to use GPU if another notebook is open and still uses GPU.

In [ ]:
# Readable model summaries, layer by layer
from torchinfo import summary

PyTorch **never** moves data to the GPU for you: you have to send both the model
(`model.to(device)`) and every batch (`x.to(device)`) explicitly. A mismatch between the device of
the model and the device of the data is one of the most common errors when starting out.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device used:", device)

# --- Speed-ups -------------------------------------------------------------
# cuDNN picks the fastest convolution algorithm for our (fixed) input size.
torch.backends.cudnn.benchmark = True

# Mixed precision: the forward/backward passes run in float16 on GPU, which is
# roughly twice as fast and halves the memory used. Disabled on CPU/MPS.
USE_AMP = (device.type == "cuda")
print("Automatic mixed precision:", USE_AMP)

# Reproducibility (relative: cuDNN remains non-deterministic by default)
SEED = 0
rd.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Dataset

The dataset used in this TP is the [$\texttt{Cats-vs-Dogs}$](https://www.kaggle.com/competitions/dogs-vs-cats) dataset used in a [Kaggle Contest](https://www.kaggle.com/c/dogs-vs-cats) which contains 25.000 images. It is a huge number when you do not have a lot of computation power.

As our goal here is to understand behaviour of algorithms and not to achieve the best performances, we have created a subsample of this dataset which are available following repository:

In [ ]:
!git clone https://plmlab.math.cnrs.fr/chevallier-teaching/datasets/cats-vs-dogs.git

### Dataset organisation

Our data are organized this way :

```
cats-vs-dogs
└───train/
│   └───cats/
│   │   │   cat.0.jpg
│   │   │   cat.1.jpg
│   │   │   ...
│   └───dogs/
│   │   │   dog.0.jpg
│   │   │   dog.1.jpg
│   │   │   ...
│   validation/
│   │   cat.1500.jpg
│   │   cat.1501.jpg
│   │   ...
│   │   dog.1500.jpg
│   │   dog.1501.jpg
│   │   ...
│   test/
│   │   cat.1000.jpg
│   │   cat.1001.jpg
│   │   ...
│   │   dog.1000.jpg
│   │   dog.1001.jpg
│   │   ...
```

> **Remark**: `torchvision.datasets.ImageFolder` could read the `train/` folder directly (one
> sub-folder per class). We keep the *dataframe* approach of the original tutorial, because the
> `validation/` and `test/` folders are flat: the label has to be read from the file name.

In [ ]:
path = "./cats-vs-dogs/"

The following cells can be used to load data in a suitable format:
* **Step 1**: Creation of $\texttt{lists}$ containing the image names of the training, validation and test sets, as well as the associated labels (0: cat, 1:dog);
* **Step 2**: Creation of $\texttt{dataframes}$ to enable data to be loaded as required.

#### Step 1: Creation of lists

In [ ]:
# Training images
train_filenames_dogs = os.listdir(path + "train/dogs")
train_filenames_cats = os.listdir(path + "train/cats")
if not os.path.exists(path + "train/train"):
    os.mkdir(path + "train/train")

path_train = path + "train/"
for filename in train_filenames_cats:
    shutil.copyfile(path_train+"cats/"+filename, path_train+"train/"+filename)
for filename in train_filenames_dogs:
    shutil.copyfile(path_train+"dogs/"+filename, path_train+"train/"+filename)

train_filenames = os.listdir(path + "train/train")
train_categories = []
for filename in train_filenames:
    category = filename.split('.')[0]
    if category == 'dog':
        train_categories.append(1)
    else:
        train_categories.append(0)


# Validation images
validation_filenames = os.listdir(path + "validation/")
validation_categories = []
for filename in validation_filenames:
    category = filename.split('.')[0]
    if category == 'dog':
        validation_categories.append(1)
    else:
        validation_categories.append(0)


# Test images
test_filenames = os.listdir(path + "test/")
test_categories = []
for filename in test_filenames:
    category = filename.split('.')[0]
    if category == 'dog':
        test_categories.append(1)
    else:
        test_categories.append(0)

#### Step 2: Creation of dataframe

In [ ]:
# Training images
total_train_df = pd.DataFrame({
    'filename': train_filenames,
    'category': train_categories
})


# Validation images
total_validation_df = pd.DataFrame({
    'filename': validation_filenames,
    'category': validation_categories
})


# Test images
test_df = pd.DataFrame({
    'filename': test_filenames,
    'category': test_categories
})


total_train_df['category'] = total_train_df['category'].astype(str)
total_validation_df['category'] = total_validation_df['category'].astype(str)
test_df['category'] = test_df['category'].astype(str)

##### <i style="color:teal">**Question:** How many training, validation and test images are available in this dataset?</i>

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/data_size.py

##### <i style="color:teal">**Question**: Is the data set balanced?</i>

In other words, do each of the train, test and validation sets contain as many cat images as dog images?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/data_ratio.py

Given the large size of this dataset, we may need to consider only a subsample of it, to speed up training.
Two parameters, $N_\text{train}$ and $N_\text{validation}$, indicate respectively the number of training and validation data to be taken into account for training.

In [ ]:
partial = 0

if partial:
    # Even numbers required
    N_train = 200   # 1000, 2000
    N_validation = 100   # 500, 1000
else:
    N_train = total_train
    N_test = total_test

In [ ]:
if N_train == total_train:
    train_df = total_train_df
    validation_df = total_validation_df
else:
    N = int(.5*N_train)
    train_df = pd.concat([total_train_df[total_train_df['category']=='1'][:N],
                          total_train_df[total_train_df['category']=='0'][:N]])
    train_df.sort_index(inplace=True)
    N = int(.5*N_validation)
    validation_df = pd.concat([total_validation_df[total_validation_df['category']=='1'][:N],
                               total_validation_df[total_validation_df['category']=='0'][:N]])
    del N

# Indices must be contiguous: a PyTorch Dataset addresses its samples by position.
train_df = train_df.reset_index(drop=True)
validation_df = validation_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

display(train_df)

### Data visualization

Images are read with [`PIL`](https://pillow.readthedocs.io/), then converted either to a
`numpy` array or directly to a `torch` tensor.

* `Image.open(...)` loads an image as a PIL image.
* `np.array(img)` produces a `(H, W, C)` `uint8` array (values in $[0, 255]$).
* `transforms.ToTensor()(img)` produces a `(C, H, W)` `float32` **tensor** already divided by 255.

Two things are worth noticing, because both will bite you later: the **channel position** moves
to the front, and the values are **automatically rescaled** to $[0, 1]$.

In [ ]:
filename = rd.choice(train_df['filename'])
img = Image.open(path + "train/train/" + filename).convert("RGB")

display(img)

plt.subplot(1, 3, 1)
plt.imshow(img)
plt.title("PIL image")

plt.subplot(1, 3, 2)
x = np.array(img)
plt.imshow(x/255, interpolation='nearest')
plt.title("Numpy image\n%s" % (x.shape,))

plt.subplot(1, 3, 3)
t = transforms.ToTensor()(img)             # (C, H, W), float in [0, 1]
plt.imshow(t.permute(1, 2, 0))             # back to (H, W, C) for matplotlib
plt.title("Torch tensor\n%s" % (tuple(t.shape),))

plt.tight_layout()
plt.show()

##### <i style="color:teal">**Question**: What are the dimensions of the $x$ array and of the $t$ tensor?</i>

To what correspond these dimensions? Why are they not in the same order?

In [ ]:
x.shape, t.shape

##### <i style="color:teal">**Question**: Does the shape of the images fluctuate?</i>

This question can be answered by producing a boxplot of widths and heights respectively.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/data_shape.py

## Pre-processing

Images have various dimensions, which is annoying because all images must have the same dimension to be used in this network.
Next, we will impose a common image size.

### `Dataset`, `transforms` and `DataLoader`

Feeding images to a network requires three distinct jobs, and PyTorch gives each of them its own
object:

1. a `Dataset` says *how to read one sample* (`__getitem__`) and *how many there are* (`__len__`);
2. a `transforms.Compose([...])` pipeline says *how to transform one image*;
3. a `DataLoader` handles *batching, shuffling and parallel loading*.

Splitting them this way is a little verbose, but very flexible: the `Dataset` below reads our
dataframes directly, and the same `DataLoader` code will work whatever we put in the `Dataset`.

> **Remark**: _Choice of $\texttt{batch-size}$ <br>
> The batch size does **not** need to divide the number of samples: the `DataLoader` simply
> yields a smaller last batch, or drops it with `drop_last=True`.

In [ ]:
class CatsDogsDataset(Dataset):
    """Reads (filename, category) pairs from one of our dataframes."""

    def __init__(self, df, directory, transform=None):
        self.df = df.reset_index(drop=True)
        self.directory = directory
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.directory, row['filename'])).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        # float label: we will use a binary cross-entropy on a single output neuron
        label = torch.tensor(float(row['category']), dtype=torch.float32)
        return img, label

We now define the transformation pipelines. Two remarks:

* `ToTensor()` already divides by 255, so there is no separate rescaling step to write.
* Pre-trained `torchvision` models expect an additional **standardisation** with the ImageNet
  mean and standard deviation. We therefore add an `imagenet_norm` switch: it stays `False` for
  the CNN we train from scratch, and will be turned `True` for the $\texttt{VGG}$ part.

In [ ]:
img_width = 150
img_height = 150
batch_size = 20

NUM_WORKERS = 2 if os.name != 'nt' else 0   # use 0 on Windows / inside some notebooks

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_transform(augment=False, imagenet_norm=False):
    steps = [transforms.Resize((img_height, img_width))]
    if augment:
        steps += [
            transforms.RandomAffine(
                degrees=40,               # random rotation, in degrees
                translate=(0.2, 0.2),     # random shift, as a fraction of width / height
                scale=(0.8, 1.2),         # random zoom in / zoom out
                shear=10,                 # random shear, in degrees
            ),
            transforms.RandomHorizontalFlip(),   # a mirrored cat is still a cat
        ]
    steps.append(transforms.ToTensor())          # /255 and (H,W,C) -> (C,H,W)
    if imagenet_norm:
        steps.append(transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return transforms.Compose(steps)


def make_loader(df, directory, augment=False, imagenet_norm=False,
                shuffle=False, batch_size=batch_size):
    dataset = CatsDogsDataset(df, directory, build_transform(augment, imagenet_norm))
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=(device.type == "cuda"),          # faster host -> GPU copies
        persistent_workers=(NUM_WORKERS > 0),        # do not respawn workers each epoch
        prefetch_factor=(4 if NUM_WORKERS > 0 else None),
    )


# Training / validation / test loaders (no augmentation yet)
train_loader = make_loader(train_df, path + "train/train/", shuffle=True)
validation_loader = make_loader(validation_df, path + "validation/")
test_loader = make_loader(test_df, path + "test/")

print("Found %d training images." % len(train_loader.dataset))
print("Found %d validation images." % len(validation_loader.dataset))
print("Found %d test images." % len(test_loader.dataset))

In [ ]:
labels = {0: 'Cat', 1: 'Dog'}
labels.get(0), labels.get(1)

##### <i style="color:teal">**Exercise** : View a sample of images.</i>

Select 9 images from the training dataset, and display them with their respective labels as titles.
The code below shows an example of how to use the loaders defined above.

**Careful**: a batch of images is a tensor of shape `(batch, channels, height, width)`; `matplotlib`
expects `(height, width, channels)`. Use `.permute(1, 2, 0)` on a single image.

In [ ]:
x_batch, y_batch = next(iter(train_loader))
print(x_batch.shape)
print(y_batch.shape)
print(y_batch)

In [ ]:
### TO BE COMPLETED ###

plt.figure(figsize=(12, 12))

[...]

plt.tight_layout()
plt.show()

In [ ]:
# %load solutions/CatsDogs/data_visualization.py

## First approach: Basic convolutional network

We will here build a classifier with a custom architecture of a convolutional network.

### Model architecture

The images have all been resized to $150\times150$. We can therefore define a convolutional neural network following this scheme:
In the first phase, this network alternates between convolution and Max Pooling layers (in order to divide the dimension of the tensors by 2 each time).

<center><img src="img/CatsDogsCNN.png" style="width:700;height:350px;"></center>
<caption><center><b> View of architecture to be implemented </b></center></caption>

<br><br>

The model we define is composed of 3 convolution blocks with the following form:
* A $\texttt{Conv2d}$ layer with $3\times3$ filters followed by a $\texttt{ReLU}$ activation.<br>
The first layer will have 32 convolution filters, the second 64, the third 96 (and the fourth 128).
* A $\texttt{MaxPool2d}$ layer with $2\times2$ window.

Followed by:
* A $\texttt{Flatten}$ layer.
* A $\texttt{Linear}$ layer with 64 (or 512) neurons and a ReLU activation function.
* A $\texttt{Dropout}$ layer with a 50% drop rate.
* A $\texttt{Linear}$ layer with 1 output neuron.

You will have built a 6-layer network, a simplified version of $\texttt{AlexNet}$.

<br>

> **Three PyTorch specificities**
>
> 1. `nn.Conv2d` needs the number of **input** channels: `nn.Conv2d(in_channels, out_channels, kernel_size)`.
> 2. Padding defaults to `0`, so every $3\times3$ convolution shrinks the map by 2 pixels. You
>    therefore have to compute the size of the flattened tensor yourself in order to size the first
>    `nn.Linear`: $150 \to 148 \to 74 \to 72 \to 36 \to 34 \to 17$, hence $96 \times 17 \times 17$.
>    (`model(torch.zeros(1, 3, 150, 150))` is a quick way to check.)
> 3. We do **not** put a sigmoid at the end. We will use `nn.BCEWithLogitsLoss`, which applies the
>    sigmoid internally in a numerically stable way. The network therefore outputs a *logit*;
>    the probability is `torch.sigmoid(logit)` and the predicted class is `logit > 0`.

In [ ]:
### TO BE COMPLETED ###

cnn_simple = nn.Sequential(
    # nn.Conv2d(...), nn.ReLU(),
    # nn.MaxPool2d(...),
    # ...
    # nn.Flatten(),    # Vectorization of the tensor to connect it to a dense layer
    # ...
).to(device)

summary(cnn_simple, input_size=(1, 3, img_height, img_width))

In [ ]:
# %load solutions/CatsDogs/CNN_model.py

#### Training

As our problem here is a two classes classifier we will use the **binary cross-entropy** loss
function, here `nn.BCEWithLogitsLoss` (binary cross-entropy applied on logits).

The training loop is written explicitly, and always follows the same five steps for each batch:

```python
optimizer.zero_grad()          # 1. reset the gradients
logits = model(x)              # 2. forward pass
loss = criterion(logits, y)    # 3. loss
loss.backward()                # 4. backward pass (gradients)
optimizer.step()               # 5. parameter update
```

The functions below wrap this loop once and for all, and return a `history` dictionary with the
keys `loss`, `accuracy`, `val_loss` and `val_accuracy`, so that every model in this notebook can
reuse them.

Note the `model.train()` / `model.eval()` calls: they switch the behaviour of `Dropout` (and of
`BatchNorm`, if any). Forgetting `model.eval()` before an evaluation is a classic bug.

<br>

> ### What the training loop needs beyond the five steps
>
> A bare loop trains, but it does not decide *when to stop*, *which weights to keep*, or *how to
> adjust the step size*. Two of these come from the standard library, one does not.
>
> **PyTorch provides the learning-rate schedules.** `torch.optim.lr_scheduler` contains
> `ReduceLROnPlateau`, `CosineAnnealingLR`, `StepLR` and a dozen others. You build one on top of
> your optimizer and call `.step()` once per epoch; nothing to reimplement. The only subtlety our
> `fit` handles for you: `ReduceLROnPlateau` needs the metric it watches
> (`scheduler.step(val_loss)`), whereas the others are called without argument
> (`scheduler.step()`).
>
> **PyTorch does not provide early stopping or best-weight checkpointing.** Core PyTorch gives
> you `torch.save` and `model.state_dict()`, but the *policy* (track the best validation loss,
> save when it improves, stop after N epochs without progress) is yours to write. 
>
> Our `fit` therefore accepts four optional arguments:
>
> * **`scheduler`** — any `torch.optim.lr_scheduler` object, stepped once per epoch.
> * **`checkpoint_path`** — after each epoch, if the validation loss improved, we save
>   `model.state_dict()` (an `OrderedDict` mapping each parameter name to its tensor). This
>   matters because **the weights at the last epoch are not the best weights**: a model that
>   overfits keeps training long after its optimum. With `restore_best_weights=True` we reload
>   that snapshot at the end.
> * **`early_stopping_patience`** — we count consecutive epochs without improvement on the
>   validation loss; past `patience`, we break out of the loop. This is the cheapest
>   regularisation there is: it costs nothing and saves compute.
> * **`train_eval_loader`** — a clean loader for an honest training metric, explained below.
>
> **Why a separate `train_eval_loader`?** The `loss`/`accuracy` accumulated during an epoch are
> computed *while the model is training*: on **augmented** images, with **`Dropout` active**, and
> with weights that change from batch to batch. They therefore measure something harder than the
> real training error, and they are what makes the training curve look worse than the validation
> curve. Passing a clean, non-augmented loader gives an honest train metric, evaluated in
> `eval()` mode at the end of the epoch, the only one comparable with the validation metric. It
> costs one extra pass over the training set per epoch, so it stays optional.

In [ ]:
import copy


def _autocast():
    """Mixed-precision context manager (a no-op when USE_AMP is False)."""
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


def _make_scaler():
    """Rescales the loss so that float16 gradients do not underflow to zero."""
    return torch.amp.GradScaler(device.type, enabled=USE_AMP)


@torch.no_grad()
def evaluate(model, loader, criterion=None, device=device):
    """Runs the model over a loader without training: returns (loss, accuracy)."""
    criterion = criterion if criterion is not None else nn.BCEWithLogitsLoss()
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        with _autocast():
            logits = model(x).squeeze(-1)
            loss = criterion(logits.float(), y)
        total_loss += loss.item() * x.size(0)
        correct += ((logits > 0).float() == y).sum().item()
        n += x.size(0)
    return total_loss / n, correct / n


def fit(model, train_loader, validation_loader, epochs, optimizer,
        criterion=None, device=device, verbose=True,
        train_eval_loader=None, scheduler=None,
        early_stopping_patience=None, checkpoint_path=None,
        restore_best_weights=True):
    """Trains `model` for `epochs` epochs and returns the history.

    train_eval_loader        : clean loader used for an honest train metric (optional)
    scheduler                : torch.optim.lr_scheduler object, stepped once per epoch
    early_stopping_patience  : stop after N epochs without improving val_loss
    checkpoint_path          : where to torch.save the best state_dict
    """
    criterion = criterion if criterion is not None else nn.BCEWithLogitsLoss()
    scaler = _make_scaler()

    history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': [], 'lr': []}
    if train_eval_loader is not None:
        history['clean_loss'], history['clean_accuracy'] = [], []

    best_val_loss, best_epoch, best_state, epochs_without_improvement = np.inf, -1, None, 0

    for epoch in range(epochs):
        t0 = time.time()

        # ---------- training pass -------------------------------------------
        model.train()
        running_loss, correct, n = 0.0, 0, 0
        for x, y in train_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)     # cheaper than zeroing the tensors
            with _autocast():
                logits = model(x).squeeze(-1)
                loss = criterion(logits.float(), y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * x.size(0)
            correct += ((logits > 0).float() == y).sum().item()
            n += x.size(0)

        history['loss'].append(running_loss / n)
        history['accuracy'].append(correct / n)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        # ---------- honest train metric (optional) ---------------------------
        if train_eval_loader is not None:
            clean_loss, clean_acc = evaluate(model, train_eval_loader, criterion, device)
            history['clean_loss'].append(clean_loss)
            history['clean_accuracy'].append(clean_acc)

        # ---------- validation ----------------------------------------------
        val_loss, val_acc = evaluate(model, validation_loader, criterion, device)
        history['val_loss'].append(val_loss)
        history['val_accuracy'].append(val_acc)

        if verbose:
            msg = ("Epoch %3d/%d - %4.1fs - loss: %.4f - accuracy: %.4f"
                   % (epoch+1, epochs, time.time()-t0,
                      history['loss'][-1], history['accuracy'][-1]))
            if train_eval_loader is not None:
                msg += " - clean_accuracy: %.4f" % clean_acc
            msg += " - val_loss: %.4f - val_accuracy: %.4f" % (val_loss, val_acc)
            print(msg)

        # ---------- ReduceLROnPlateau / LR schedule --------------------------
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        # ---------- ModelCheckpoint + EarlyStopping --------------------------
        if val_loss < best_val_loss:
            best_val_loss, best_epoch = val_loss, epoch
            epochs_without_improvement = 0
            best_state = copy.deepcopy(model.state_dict())
            if checkpoint_path is not None:
                torch.save(best_state, checkpoint_path)
        else:
            epochs_without_improvement += 1
            if (early_stopping_patience is not None
                    and epochs_without_improvement >= early_stopping_patience):
                print("Early stopping at epoch %d (no improvement for %d epochs)."
                      % (epoch+1, early_stopping_patience))
                break

    if restore_best_weights and best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print("Restored the weights of epoch %d (val_loss = %.4f)."
                  % (best_epoch+1, best_val_loss))

    return history


@torch.no_grad()
def predict(model, loader, device=device):
    """Returns (predicted probabilities, true labels) for a whole loader."""
    model.eval()
    probs, ys = [], []
    for x, y in loader:
        with _autocast():
            logits = model(x.to(device, non_blocking=True)).squeeze(-1)
        probs.append(torch.sigmoid(logits.float()).cpu())
        ys.append(y)
    return torch.cat(probs).numpy(), torch.cat(ys).numpy()


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print("Total params: %s | Trainable: %s | Non-trainable: %s"
          % (format(total, ','), format(trainable, ','), format(total-trainable, ',')))
    return total, trainable

Finally, a small bookkeeping helper. Every model we build will be scored on the three
sets and its results stored in a global `results` dictionary; the last exercise of the notebook
will turn it into a comparison table. Note that the **test set is used only for reporting**,
never to choose a model or a hyper-parameter.

In [ ]:
results = {}


def record(name, model, loaders, t_learning=None, criterion=None, verbose=True):
    """Scores a model on (train, validation, test) and stores everything in `results`."""
    train_l, val_l, test_l = loaders

    t0 = time.time()
    tr_loss, tr_acc = evaluate(model, train_l, criterion)
    va_loss, va_acc = evaluate(model, val_l, criterion)
    te_loss, te_acc = evaluate(model, test_l, criterion)
    t_prediction = time.time() - t0

    results[name] = {
        'train_loss': tr_loss, 'train_accuracy': tr_acc,
        'val_loss': va_loss, 'val_accuracy': va_acc,
        'test_loss': te_loss, 'test_accuracy': te_acc,
        'learning_time': t_learning,
        'prediction_time': t_prediction,
    }

    if verbose:
        print("[%s]" % name)
        print("  Train accuracy      : %.4f" % tr_acc)
        print("  Validation accuracy : %.4f" % va_acc)
        print("  Test accuracy       : %.4f" % te_acc)
        print("  Time Prediction     : %.2f seconds" % t_prediction)
    return results[name]

* $\texttt{batch\_size}$: it has already been fixed when building the `DataLoader`.

* $\texttt{epochs}$: start with a small number (5-10) in order to check that computing time is reasonable.

* the **optimizer** replaces the `optimizer=` argument of `compile`. It must be given the
  parameters it is allowed to update: this is how PyTorch expresses "trainable weights".

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(cnn_simple.parameters(), lr=3e-4)

In [ ]:
epochs = 30

t_learning_cnn_simple = time.time()
cnn_simple_history = fit(
    cnn_simple,
    train_loader,
    validation_loader,
    epochs=epochs,
    optimizer=optimizer,
    criterion=criterion,
)
t_learning_cnn_simple = time.time() - t_learning_cnn_simple

print("Learning time for %d epochs : %d seconds" % (epochs, t_learning_cnn_simple))

#### Analysis of results

In [ ]:
record("CNN", cnn_simple,
       (train_loader, validation_loader, test_loader),
       t_learning=t_learning_cnn_simple)

##### <i style="color:teal">**Exercise**: Visualize the evolution of metrics during training.</i>

Write a function to display the evolution of metrics during training, on the training and validation sets. Display accuracy and loss on separate figures.

Our `fit` returns a plain `dict`: the curves you need are `history['accuracy']`,
`history['val_accuracy']`, `history['loss']` and `history['val_loss']`.

In [ ]:
### TO BE COMPLETED ###

def plot_training_analysis():
    [...]

In [ ]:
# %load solutions/CatsDogs/plot_training_analysis.py

In [ ]:
plot_training_analysis(cnn_simple_history)

##### <i style="color:teal">**Question** : What phenomenon are you dealing with?</i>

### Overfitting correction

With the previous network, we were in a typical **overfitting** situation. This is a classic problem when working with small databases in deep learning.
Actually, the network you have created normally contains several million parameters (if you've followed the instructions). The problem you are trying to solve during training is establishing 1.8 million parameters with just 2,000 examples: that is too few!

To limit this overfitting, we can apply regularization techniques. In image processing, one of the most commonly used techniques is **data augmentation**.

In PyTorch, data augmentation is just **more transforms** inserted in the pipeline, before
`ToTensor()`. They are applied *on the fly*, differently at each epoch, since `__getitem__` is
called again every time. Have a look at the [documentation](https://pytorch.org/vision/stable/transforms.html)
to find out what the parameters below correspond to.

> **Watch the borders**: a rotation or a translation leaves empty pixels along the edges.
> `RandomAffine` fills them with a constant (`fill=0`, i.e. black) by default; look at the images
> below and ask yourself whether those black corners could themselves become a feature the
> network learns from.

In [ ]:
transforms.RandomAffine?

In [ ]:
train_loader_augmented = make_loader(
    train_df, path + "train/train/",
    augment=True,      # rotation, translation, zoom, shear, horizontal flip
    shuffle=True,
)

print(train_loader_augmented.dataset.transform)

##### <i style="color:teal">**Question**: Why do we apply different transformations for learning and validation?</i>

Indeed, we do not define a new augmented loader for the validation set.

In the next cell, you can view images that have passed through our data augmentation loop. Observe how missing values in the images (for example, in the case of rotation) are filled in.

In [ ]:
plt.figure(figsize=(12, 12))

example_x, example_y = next(iter(train_loader_augmented))

for i in range(9):
    plt.subplot(3, 3, i+1)
    plt.imshow(example_x[i].permute(1, 2, 0))
    plt.title(labels.get(int(example_y[i])))
    plt.axis('off')

plt.tight_layout()
plt.show()

#### Training

We can now rebuild our model and start training again.

**Careful**: an optimizer holds a reference to the parameter tensors it was given. Rebuilding
the model creates *new* tensors, so the previous `optimizer` would keep updating the old,
discarded ones, silently, without any error. A new optimizer must be created every time the model
is rebuilt.

This is also the right moment to switch on the callbacks described above: a `ReduceLROnPlateau`
scheduler, an early stopping with a patience of 8 epochs, a checkpoint on the best validation
loss, and a clean `train_eval_loader` so that the training curve is honest.

In [ ]:
### TO BE COMPLETED ###

cnn_simple = nn.Sequential(
    [...]
).to(device)

summary(cnn_simple, input_size=(1, 3, img_height, img_width))

# --- #

### TO BE COMPLETED ###
epochs = 10

optimizer = ...

t_learning_cnn_simple_augmented = ...
cnn_simple_augmented_history = fit(
    cnn_simple,
    train_loader_augmented,
    validation_loader,
    epochs = epochs,
    optimizer = optimizer,
)

print("Learning time for %d epochs : %d seconds" % (epochs, t_learning_cnn_simple_augmented))

In [ ]:
# %load solutions/CatsDogs/CNN_model.py

In [ ]:
# %load solutions/CatsDogs/CNN_train.py

#### Analysis of results

In [ ]:
record("CNN + augmentation", cnn_simple,
       (train_loader, validation_loader, test_loader),
       t_learning=t_learning_cnn_simple_augmented)

In [ ]:
plot_training_analysis(cnn_simple_augmented_history)

##### <i style="color:teal">**Exercise**: Compare the three training curves.</i>

The blue dashed curve (`loss` / `accuracy`) is accumulated *during* the epoch, on augmented
images, with `Dropout` active and with weights that keep changing. The red dotted curve
(`clean_loss` / `clean_accuracy`) is the same training set, but clean and in `eval()` mode.

* Which of the two is comparable with the green validation curve, and why?
* Which of the two would you use to diagnose overfitting?
* The blue curve is often *below* the green one at the beginning of training. Is this normal?
* Compare the values printed at the last epoch with those returned by `record` above: which
  quantity did the early stopping actually restore?

The curves clearly show that overfitting has been limited. Note also, and this is important, that training is slower: the model takes longer to correctly predict the training set. This is to be expected, as we have somehow "complicated the problem" by introducing all these deformations to our images.
This form of "data-driven" regularization is in addition to the other methods such as L1/L2 regularization of network weights (in PyTorch: the `weight_decay` argument of the optimizer) and Dropout.

One should now achieve around 75% accuracy on the validation set, which is good but not completely satisfactory: to continue improving, you will probably need to train longer but also have more data at your disposal.

Another solution is to use **Transfer Learning**.

## Pre-trained Network

We have seen above that the complexity of the data makes it difficult to build quickly an efficient classifier from scratch even with an elaborate method as a convolutional network.
One reason why our results were disappointing is that the first layers of our convolutional network, which are supposed to detect features useful for discriminating between dogs and cats, didn't learn sufficiently general filters from the 2000 training images. So, even if these filters are relevant for the 2000 training images, there is little chance that they will work well for generalization on new data.

This is why we want to reuse a **pre-trained network** on a large database, enabling us to detect features that will generalize better to new data.

The figure below represents a $\texttt{VGG-16}$. This model is composed of _5 convolutional blocks_ which allows to build features on the images. The last block is a _fully connected block_.

<center><img src="https://blog.keras.io/img/imgclf/vgg16_original.png" style="height:700px;"></center>
<caption><center><b> VGG-16 </b></center></caption>

Here is our two-stage strategy :
1. **Features map**: We will send our data through the 5 convolutional blocks in order to build features.
2. **MLP classifier**: We will build our own MLP classifier designed to solve our Cats-vs-Dogs problem, and we will train it on the features built on the first step.

### Step 1 : Build features

In [ ]:
from torchvision.models import vgg16, VGG16_Weights

#### Download the weights of the 5 blocks convolutional layer.

We will now download the weights of a VGG16 model that has been learned on the [$\texttt{ImageNet}$](http://www.image-net.org) dataset, which is composed of millions of images for 1000 categories.

If it's the first time you use these weights, they will be downloaded automatically and saved in
`"~/.cache/torch/hub/checkpoints"`.

In `torchvision`, a `vgg16` model is split into three attributes: `.features` (the 5 convolutional
blocks), `.avgpool` and `.classifier` (the dense head that outputs the 1000 ImageNet classes).
Keeping only `.features` gives us the feature extractor and drops the head, which is exactly what
we want.

In [ ]:
conv_base = vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features.to(device)

summary(conv_base, input_size=(1, 3, img_height, img_width))

Pre-trained `torchvision` models were trained on images standardised with the ImageNet
statistics. We therefore rebuild our loaders with `imagenet_norm=True`.

Note also `shuffle=False` for the training loader here: we want the features and the labels to
stay aligned, and the `DataLoader` gives us the labels directly, so there is never any need to go
back to the dataframe. Keep this in mind, an exercise below comes back to it.

In [ ]:
train_loader_vgg = make_loader(train_df, path + "train/train/", imagenet_norm=True, shuffle=False)
validation_loader_vgg = make_loader(validation_df, path + "validation/", imagenet_norm=True)
test_loader_vgg = make_loader(test_df, path + "test/", imagenet_norm=True)

train_loader_vgg_augmented = make_loader(
    train_df, path + "train/train/", augment=True, imagenet_norm=True, shuffle=True)

#### Building features

The structure of the $\texttt{VGG}$ network summarized above shows that the output tensor is of
dimension $512\times4\times4$ (channels first!), _i.e._ the network predicts features of dimension
$512\times4\times4$ from an image of size $150\times150$.

Therefore, we must take care to vectorize this output so that we can pass it through a dense
network. `tensor.flatten(1)` flattens every dimension but the batch one.

In [ ]:
@torch.no_grad()
def extract_features(conv_base, loader, device=device):
    """Runs the frozen convolutional base once and returns (features, labels)."""
    conv_base.eval()
    feats, ys = [], []
    for x, y in loader:
        f = conv_base(x.to(device))
        feats.append(f.flatten(1).cpu())   # (B, 512, 4, 4) -> (B, 8192)
        ys.append(y)
    return torch.cat(feats), torch.cat(ys)


# The convolutional base is frozen, so its output never changes: computing it once and
# caching it on disk saves a full VGG forward pass every time the notebook is re-run.
FEATURES_CACHE = "vgg16_features.pt"

if os.path.exists(FEATURES_CACHE):
    cache = torch.load(FEATURES_CACHE)
    train_features, y_train = cache['train_features'], cache['y_train']
    validation_features, y_validation = cache['validation_features'], cache['y_validation']
    test_features, y_test_feat = cache['test_features'], cache['y_test']
    print("Features reloaded from", FEATURES_CACHE)
else:
    t0 = time.time()
    train_features, y_train = extract_features(conv_base, train_loader_vgg)
    validation_features, y_validation = extract_features(conv_base, validation_loader_vgg)
    test_features, y_test_feat = extract_features(conv_base, test_loader_vgg)
    print("Features extracted in %.1f s" % (time.time() - t0))
    torch.save({'train_features': train_features, 'y_train': y_train,
                'validation_features': validation_features, 'y_validation': y_validation,
                'test_features': test_features, 'y_test': y_test_feat}, FEATURES_CACHE)

print(train_features.shape, validation_features.shape, test_features.shape)

The labels come straight out of the loaders, alongside the features they belong to, so there
is no risk of a mismatch between the two. We wrap them in a `TensorDataset` in order to reuse our
`fit` function unchanged.

In [ ]:
# Bigger batches: these are tiny vectors, not images
train_features_loader = DataLoader(TensorDataset(train_features, y_train),
                                   batch_size=64, shuffle=True)
train_features_eval_loader = DataLoader(TensorDataset(train_features, y_train), batch_size=256)
validation_features_loader = DataLoader(TensorDataset(validation_features, y_validation),
                                        batch_size=256)
test_features_loader = DataLoader(TensorDataset(test_features, y_test_feat), batch_size=256)

### Step 2 : Building our classifier on top of features

We can now define a simple neural network that will work directly on the features predicted by $\texttt{VGG}$.

##### <i style="color:teal">**Exercise**: Write this classifier.</i>

In [ ]:
### TO BE COMPLETED ###

vgg_mlp = nn.Sequential(
    [...]
).to(device)

summary(vgg_mlp, input_size=(1, train_features.shape[1]))

# --- #

### TO BE COMPLETED ###
epochs = 10

optimizer = ...

t_learning_vgg_mlp = ...
vgg_mlp_history = fit(
    vgg_mlp,
    train_features_loader,
    validation_features_loader,
    epochs = epochs,
    optimizer = optimizer,
)

print("Learning time for %d epochs : %d seconds" % (epochs, t_learning_vgg_mlp))

In [ ]:
# %load solutions/CatsDogs/MLP_model.py

In [ ]:
# %load solutions/CatsDogs/MLP_train.py

#### Analysis of results

In [ ]:
record("VGG features + MLP", vgg_mlp,
       (train_features_eval_loader, validation_features_loader, test_features_loader),
       t_learning=t_learning_vgg_mlp)

In [ ]:
plot_training_analysis(vgg_mlp_history)

The validation accuracy jumps to roughly 90%, far above everything we obtained by training
from scratch, with a model of a few thousand parameters trained in seconds. The gap between the
training and validation curves shows that the classifier still overfits, but the features
themselves generalise very well.

<br>

##### <i style="color:teal">**Exercise**: the shuffling trap.</i>

Building `train_loader_vgg`, we insisted on `shuffle=False`, and we took the labels returned by
the loader itself rather than reading them from `train_df`. Let us see what happens otherwise.

Redo the feature extraction with a loader built with `shuffle=True`, but pair those features with
the labels **read from the dataframe** (`train_df['category']`), which is the natural thing to do
when one thinks of the loader as "just a way to read the images". Train the very same MLP on the
result and plot the curves.

* What happens to the training accuracy? To the validation accuracy?
* Would you have diagnosed the problem as *overfitting* if you had only seen these curves?
* Where else in this notebook would the same bug be invisible but just as damaging?
* How could you have detected it *before* training anything?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/shuffle_trap.py

The training accuracy climbs steadily while the validation accuracy stays glued to 0.5,
_i.e._ the score of a coin flip. The network is not overfitting: it is **memorising pure noise**,
because each feature vector has been paired with the label of a different image. There is nothing
to generalise, so validation performance cannot move.

The lesson is worth more than the model: a training curve that goes up while the validation curve
stays at chance level is almost never "too little data" or "too much capacity": it is a **data
pipeline bug**. And the sanity check costs one line: display a handful of images together with
the label you are about to feed the network.

Note in passing that the same bug would be invisible at prediction time, where a shuffled test
loader would silently misalign `test_prediction[i]` and `test_df['filename'][i]`.

## Transfer Learning combined with Data Augmentation

### Training

Our features were computed once, on clean images: the classifier therefore sees the same 2000
vectors at every epoch, which is exactly why it overfits. To bring back data augmentation, we now
connect our small network to the end of the $\texttt{VGG}$ convolutional base, so that the
features are recomputed on the augmented images at each iteration.

We start by creating a new model based on $\texttt{VGG}$'s convolutional base, to which we add a dense layer and our output layer.

An `nn.Sequential` can contain another module, so `conv_base` is simply plugged in as a first
"layer": a whole 13-layer network becomes a single building block of a larger one.

In [ ]:
vgg_combined = nn.Sequential(
    conv_base,
    nn.Flatten(),
    nn.Linear(512*4*4, 256), nn.ReLU(),
    nn.Linear(256, 1),
).to(device)

summary(vgg_combined, input_size=(1, 3, img_height, img_width))

**Caution**: It is important to avoid training $\texttt{VGG}$'s convolutional base! We do not want to override the good features of VGG that we are trying to reuse! The network would also have a large number of parameters, which is precisely what we want to avoid.

Freezing is expressed by the `requires_grad` flag carried by each *tensor* of parameters. Setting
it to `False` tells autograd not to compute a gradient for that tensor, so it can no longer be
updated during training.

Two things to keep in mind:
* the optimizer must be built (or rebuilt) **after** freezing, and given only the parameters that
  require a gradient;
* freezing is *not* the same as `eval()` mode. Here `VGG` contains no `BatchNorm`, so it does not
  matter, but with a `ResNet` you would also want to keep the frozen base in `eval()` mode.

In [ ]:
for p in conv_base.parameters():
    p.requires_grad = False

count_parameters(vgg_combined)

Look at the number of weights: the number of trainable weights is now 2 million, compared with 16 million previously; we are only going to train the weights of our dense layer and the output layer.

In [ ]:
epochs = 5

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    [p for p in vgg_combined.parameters() if p.requires_grad], lr=3e-4)

t_learning_vgg_combined = time.time()
vgg_combined_history = fit(
    vgg_combined,
    train_loader_vgg_augmented,
    validation_loader_vgg,
    epochs=epochs,
    optimizer=optimizer,
    criterion=criterion,
    train_eval_loader=train_loader_vgg,   # honest train metric (see the note above)
)
t_learning_vgg_combined = time.time() - t_learning_vgg_combined

print("Learning time for %d epochs : %d seconds" % (epochs, t_learning_vgg_combined))

Training is much slower! We have to generate the augmented data, and feed it through the VGG layers at each gradient iteration. This takes time.

> **On the cost of `train_eval_loader`.** We pass it to every training from here on, so the red
> dotted curve is available on all the transfer-learning models. It is not free: each epoch now
> ends with one extra forward pass over the 2000 training images through the whole of
> $\texttt{VGG}$, which lengthens an epoch by roughly a third. We pay it deliberately, because
> from this point on the claims we want to make are precisely about the gap between training and
> validation, and the blue curve (measured on augmented images, mid-optimisation) cannot
> support them. If you need the time back, evaluate on a fixed random subset of the training set
> rather than dropping the measurement: a few hundred images already estimate the training
> accuracy to within a percent or so.
>
> Note also that `vgg_combined` contains no $\texttt{Dropout}$, unlike the CNN we built by hand.
> The gap between the blue and the red curve here therefore measures the effect of **data
> augmentation alone**, which makes it easier to read.

### Analysis of results

In [ ]:
# Note the clean (non-augmented) train loader: see the exercise on honest metrics
record("VGG + augmentation (frozen)", vgg_combined,
       (train_loader_vgg, validation_loader_vgg, test_loader_vgg),
       t_learning=t_learning_vgg_combined)

In [ ]:
plot_training_analysis(vgg_combined_history)

On the other hand, overfitting has been limited, which was the aim. This considerably improves results.

## Fine Tuning

We have notably increased the performances of our model with a model that is really efficient. We can continue to try to improve our results by modifying the small MLP classifier network we build.

But to really improve our model, it would be nice to also change the weights of the previous layers in order to make them fit our problem. This is possible and called **Fine Tuning**.
To do this, we are going to start again from the network we just trained, but will unlock the training of all or part of the weights of the entire network.
<br><br>

> **WARNING**: It is important to choose a very low learning rate so as not to wipe out the benefits of previous training sessions.
The aim is simply to evolve the network parameters "at the margin", and this can only be done after the first *transfer learning* step.

<center><img src="https://blog.keras.io/img/imgclf/vgg16_modified.png" style="height:700px;"></center>
<caption><center><b> VGG-16 </b></center></caption>

### Training

We will start by fine-tuning only the last block of convolution. To that aim, we reactivate the
gradients on the whole $\texttt{VGG}$ convolutional base, then freeze again everything up to the
last block.

In `torchvision`, `vgg16.features` is an `nn.Sequential` of 31 layers; the fifth (and last)
convolutional block starts at index **24**. Run the cell below to convince yourself.

In [ ]:
for i, layer in enumerate(conv_base):
    print(i, layer)

In [ ]:
for p in conv_base.parameters():
    p.requires_grad = True

for layer in conv_base[:24]:              # everything before block 5 stays frozen
    for p in layer.parameters():
        p.requires_grad = False

count_parameters(vgg_combined)

#### Differentiated learning rates (*discriminative fine-tuning*)

Up to now we used a single learning rate for the whole model, which forces an uncomfortable
compromise: it must be small enough not to destroy the ImageNet filters, yet large enough for our
freshly initialised head to keep learning. There is no reason to accept that compromise: the
optimizer accepts a list of **parameter groups**, each with its own settings.

```python
optimizer = torch.optim.Adam([
    {'params': base_params, 'lr': 1e-5},   # pre-trained: nudge gently
    {'params': head_params, 'lr': 1e-4},   # our head: keep learning
])
```

The rule of thumb is that the deeper a layer sits in the network, the more general (and the more
worth preserving) its filters are, so the smaller its learning rate should be. Some libraries
push this further and assign a geometrically decreasing rate to every block.

Here `vgg_combined[0]` is the convolutional base and `vgg_combined[1:]` is our classification
head.

In [ ]:
epochs = 5

criterion = nn.BCEWithLogitsLoss()

# The optimizer MUST be rebuilt: it captured the previous list of trainable parameters
base_params = [p for p in vgg_combined[0].parameters() if p.requires_grad]
head_params = [p for p in vgg_combined[1:].parameters() if p.requires_grad]

optimizer = torch.optim.Adam([
    {'params': base_params, 'lr': 1e-5},   # VGG block 5: very small steps
    {'params': head_params, 'lr': 1e-4},   # our own layers: ten times larger
])
print("Base parameters: %d tensors | Head parameters: %d tensors"
      % (len(base_params), len(head_params)))

t_learning_vgg_combined_tuned = time.time()
vgg_combined_tuned_history = fit(
    vgg_combined,
    train_loader_vgg_augmented,
    validation_loader_vgg,
    epochs=epochs,
    optimizer=optimizer,
    criterion=criterion,
    train_eval_loader=train_loader_vgg,   # honest train metric (see the note above)
)
t_learning_vgg_combined_tuned = time.time() - t_learning_vgg_combined_tuned

print("Learning time for %d epochs : %d seconds" % (epochs, t_learning_vgg_combined_tuned))

### Analysis of results

In [ ]:
record("VGG fine-tuned (block 5)", vgg_combined,
       (train_loader_vgg, validation_loader_vgg, test_loader_vgg),
       t_learning=t_learning_vgg_combined_tuned)

In [ ]:
plot_training_analysis(vgg_combined_tuned_history)

### Training again

We decide to continue fine-tuning the entire convolutional base, with an even smaller step size.

In [ ]:
for p in conv_base.parameters():
    p.requires_grad = True

count_parameters(vgg_combined)

In [ ]:
epochs = 5

criterion = nn.BCEWithLogitsLoss()

# Same idea, one notch lower: the whole base is now trainable
base_params = [p for p in vgg_combined[0].parameters() if p.requires_grad]
head_params = [p for p in vgg_combined[1:].parameters() if p.requires_grad]

optimizer = torch.optim.Adam([
    {'params': base_params, 'lr': 1e-6},
    {'params': head_params, 'lr': 1e-5},
])

t_learning_vgg_combined_tuned2 = time.time()
vgg_combined_tuned_history2 = fit(
    vgg_combined,
    train_loader_vgg_augmented,
    validation_loader_vgg,
    epochs=epochs,
    optimizer=optimizer,
    criterion=criterion,
    train_eval_loader=train_loader_vgg,   # honest train metric (see the note above)
)
t_learning_vgg_combined_tuned2 = time.time() - t_learning_vgg_combined_tuned2

print("Learning time for %d epochs : %d seconds" % (epochs, t_learning_vgg_combined_tuned2))

### Analysis of results

In [ ]:
record("VGG fine-tuned (all)", vgg_combined,
       (train_loader_vgg, validation_loader_vgg, test_loader_vgg),
       t_learning=t_learning_vgg_combined_tuned2)

In [ ]:
plot_training_analysis(vgg_combined_tuned_history2)

## Prediction on Kaggle Dataset

Let's see now how our trained model performs on the kaggle real test dataset ($\texttt{cats-vs-dogs/test}$)

##### <i style="color:teal">**Exercise**: Apply the model to this dataset and display results on a sample to check it performs well</i>

Predicted labels will be displayed as chart titles, colored green if the prediction is correct and red if not.

*Hint*: `test_loader_vgg` was built with `shuffle=False`, so the predictions come back in the same
order as the rows of `test_df`. For the display, reload the raw images with `PIL` (the tensors of
the loader are standardised, hence not displayable as-is).

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/test_kaggle.py

### Confusion matrix and worst mistakes

Nine images drawn at random are reassuring but not very informative: with 92% accuracy, most of
them are correct. Two displays are far more useful.

The **confusion matrix** splits the errors by class and answers a question the accuracy hides:
does the model confuse cats with dogs as often as the other way round? The threshold at 0.5 is
arbitrary, and moving it trades one kind of error for the other.

The **worst mistakes** are the images the model got wrong *with the highest confidence*. They are
the most informative images of the whole dataset: they usually reveal either a genuine weakness
of the model (unusual pose, several animals, heavy occlusion) or a mislabelled image.

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/confusion_matrix.py

## Global Average Pooling

Actually, we no longer really use the $\texttt{Flatten}$ layer to bridge the gap between convolutional and dense layers, but rather a **Global Average Pooling** layer. Try to understand what this layer does and modify the network built on top of $\texttt{VGG}$ accordingly.

PyTorch has no layer under that exact name: the operation is expressed as
`nn.AdaptiveAvgPool2d(1)` (average over the whole spatial map, whatever its size) followed by
`nn.Flatten()`. The $512\times4\times4$ tensor becomes a vector of size $512$ instead of $8192$.

In [ ]:
nn.AdaptiveAvgPool2d?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/GlobalAveragePooling_model.py

In [ ]:
# %load solutions/CatsDogs/GlobalAveragePooling_train.py

In [ ]:
# %load solutions/CatsDogs/GlobalAveragePooling_tune.py

In [ ]:
plot_training_analysis(history_combined_average)
plot_training_analysis(history_combined_average2)

##### <i style="color:teal">**Question**: compare with the $\texttt{Flatten}$ version.</i>

The head went from $8192 \times 256 \approx 2.1$M parameters down to $512 \times 256 \approx 131$k,
a factor of 16. Is the test accuracy 16 times worse? What does that tell you about where the
useful information actually lives in a convolutional network?

## Comparing every model

Throughout the notebook, `record` has been storing the scores and timings of each model in the
`results` dictionary. Time to use it.

##### <i style="color:teal">**Exercise**: build the summary table.</i>

Turn `results` into a `DataFrame` (one row per model), sorted by test accuracy, then plot the
train / validation / test accuracies side by side.

Then comment:
* Which model would you actually ship, taking the training time into account?
* Is the ranking on the validation set the same as on the test set? If not, why is that expected?
* We used the validation set to early-stop and to choose between architectures. What does that
  imply about the validation accuracy as an estimate of future performance?

In [ ]:
### TO BE COMPLETED ###

[...]

In [ ]:
# %load solutions/CatsDogs/results_summary.py

## Exercise

`torchvision.models` provides a lot of pre-trained models, listed with
`torchvision.models.list_models()`:

* `resnet18`, `resnet50`
* `vgg16`, `vgg19`
* `inception_v3`
* `mobilenet_v3_small`
* `efficientnet_b0`
* `convnext_tiny`
* `vit_b_16`
* ...

Some have a much more complex architecture, and they are **not** all split the same way:
`resnet50` has no `.features` attribute (use `nn.Sequential(*list(model.children())[:-1])`, or
replace `model.fc` by your own head), `inception_v3` expects $299\times299$ inputs, etc.

A generic and robust way to get the feature extractor of any `torchvision` model is:

```python
from torchvision.models.feature_extraction import create_feature_extractor
```

Do not forget to use the right normalisation for each model:
`weights.transforms()` returns the exact pre-processing pipeline it was trained with.

In [ ]:
weights = VGG16_Weights.IMAGENET1K_V1
print(weights.transforms())

##### <i style="color:teal">**Exercise**: Restart the TP by using a different pre-trained model and apply the required modifications.</i>